# TruthAnchor Demo

This notebook runs the TruthAnchor pipeline step by step on one example setup:

- model: `meta-llama/Llama-3.2-3B-Instruct`
- dataset: `trivia`


In [ ]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,4"

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT / "src"))
elif (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT / "src"))
else:
    raise RuntimeError("Could not find the project src/ directory.")


In [ ]:
import csv
import numpy as np
from sklearn.model_selection import train_test_split

from truthanchor import (
    compute_uncertainty_scores,
    evaluate_saved_mappers,
    generate_responses,
    train_mappers,
)
from truthanchor.utils.io import load_jsonl, load_npz_dict
from truthanchor.utils.methods import METHOD_INFO, METHODS_PLOT
from truthanchor.utils.paths import artifact_dir
from truthanchor.utils.visualization import (
    plot_auc_comparison,
    plot_calibration_diagram,
    plot_ece_comparison,
)


In [ ]:
model_name = "meta-llama/Llama-3.2-3B-Instruct"
dataset_name = "trivia"
output_root = "outputs"
max_new_tokens = 128
num_samples = 5
batch_size = 1
data_portion = 1.0
rank_weight = 1.0
ue_methods = list(METHOD_INFO)

out_dir = artifact_dir(output_root, dataset_name, model_name)
results_dir = out_dir / "mapper_eval"
scores_dir = results_dir / "scores"
results_dir.mkdir(parents=True, exist_ok=True)
scores_dir.mkdir(parents=True, exist_ok=True)

out_dir

## Step 1: Generate Responses

In [ ]:
generate_responses(
    model_name=model_name,
    dataset_name=dataset_name,
    max_new_tokens=max_new_tokens,
    data_portion=data_portion,
    num_samples=num_samples,
    save=True,
    output_root=output_root,
    batch_size=batch_size,
)

print(out_dir / f"responses_{dataset_name}.jsonl")
print(out_dir / "generation_results.npz")

## Step 2: Compute Uncertainty Scores

In [ ]:
compute_uncertainty_scores(
    model_name=model_name,
    dataset_name=dataset_name,
    output_root=output_root,
)

print(out_dir / "uncertainty_scores.npz")

## Step 3: Load Prepared Data

In [ ]:
uncertainty_data = load_npz_dict(out_dir / "uncertainty_scores.npz")
texts = np.asarray([row["prompt"] for row in load_jsonl(out_dir / f"responses_{dataset_name}.jsonl")])
labels = 1 - np.asarray(uncertainty_data["labels"])

print("num examples:", len(labels))
print("available score keys:", sorted(k for k in uncertainty_data.keys() if k != "labels"))

## Step 4: Train/Evaluate Per Uncertainty Method

In [ ]:
orig_eces = []
mapped_eces = []
orig_aucs = []
mapped_aucs = []
plotted_methods = []
rows = []

indices = np.arange(len(labels))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

for method in ue_methods:
    key, higher_worse = METHOD_INFO[method]
    X = np.asarray(uncertainty_data[key]).reshape(-1, 1)
    if higher_worse:
        X = -X

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = labels[train_idx], labels[test_idx]

    mapper, _, val_report = train_mappers(
        X_train,
        y_train,
        rank_weight=rank_weight,
        use_cue=False,
    )
    test_report = evaluate_saved_mappers(mapper, X_test, y_test)

    print(
        f"{method}: test mapped AUROC = {test_report['auroc_mapped']:.4f}, "
        f"ECE = {test_report['ece_mapped']:.4f}, PRR = {test_report['prr_mapped']:.4f}; "
        f"test orig AUROC = {test_report['auroc_orig']:.4f}, "
        f"ECE = {test_report['ece_orig']:.4f}, PRR = {test_report['prr_orig']:.4f}"
    )

    np.savez_compressed(
        scores_dir / f"{method}_validation_scores.npz",
        labels=val_report["labels"],
        raw_scores=val_report["raw_scores"],
        mapped_scores=val_report["mapped_scores"],
    )
    np.savez_compressed(
        scores_dir / f"{method}_test_scores.npz",
        labels=test_report["labels"],
        raw_scores=test_report["raw_scores"],
        mapped_scores=test_report["mapped_scores"],
    )

    rows.append(
        {
            "method": method,
            "val_auroc_orig": val_report["auroc_orig"],
            "val_ece_orig": val_report["ece_orig"],
            "val_prr_orig": val_report["prr_orig"],
            "val_auroc_mapped": val_report["auroc_mapped"],
            "val_ece_mapped": val_report["ece_mapped"],
            "val_prr_mapped": val_report["prr_mapped"],
            "test_auroc_orig": test_report["auroc_orig"],
            "test_ece_orig": test_report["ece_orig"],
            "test_prr_orig": test_report["prr_orig"],
            "test_auroc_mapped": test_report["auroc_mapped"],
            "test_ece_mapped": test_report["ece_mapped"],
            "test_prr_mapped": test_report["prr_mapped"],
        }
    )

    if method in METHODS_PLOT:
        plotted_methods.append(METHODS_PLOT[method])
        orig_eces.append(test_report["ece_orig"])
        mapped_eces.append(test_report["ece_mapped"])
        orig_aucs.append(test_report["auroc_orig"])
        mapped_aucs.append(test_report["auroc_mapped"])


## Step 5: Save Result Table

In [ ]:
results_path = results_dir / "results.csv"
with results_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

results_path

## Step 6: Create Plots

In [ ]:
plot_ece_comparison(dataset_name, plotted_methods, orig_eces, mapped_eces, results_dir)
plot_auc_comparison(dataset_name, plotted_methods, orig_aucs, mapped_aucs, results_dir)

for row in rows:
    method = row["method"]
    if method not in METHODS_PLOT:
        continue
    test_scores = np.load(scores_dir / f"{method}_test_scores.npz")
    plot_calibration_diagram(
        test_scores["raw_scores"],
        test_scores["labels"],
        row["test_auroc_orig"],
        row["test_ece_orig"],
        METHODS_PLOT[method],
        results_dir,
        anchored=0,
    )
    plot_calibration_diagram(
        test_scores["mapped_scores"],
        test_scores["labels"],
        row["test_auroc_mapped"],
        row["test_ece_mapped"],
        METHODS_PLOT[method],
        results_dir,
        anchored=1,
    )

print(results_dir)

## Outputs

The notebook writes:

- generation artifacts to `outputs/trivia/meta-llama--Llama-3.2-3B-Instruct/`
- mapper evaluation CSV to `mapper_eval/results.csv`
- validation/test score files to `mapper_eval/scores/`
- plots to `mapper_eval/`
